# Retrieval-Augmented Generation (RAG)

## Convert document to text

In [1]:
from langchain_community.document_loaders import TextLoader

loader = TextLoader("./test.txt")
loader.load()

[Document(metadata={'source': './test.txt'}, page_content="Bonjour, aujourd'hui est d'un beau jour avec le gros soleil.\nLes choses les plus mystÃ¨re du monde sont des mystÃ¨res.\nvous saurez ces secrets quand vous les dÃ©couvrez.")]

In [2]:
from langchain_community.document_loaders import WebBaseLoader

loader = WebBaseLoader("https://www.langchain.com/")
loader.load()

USER_AGENT environment variable not set, consider setting it to identify your requests.


[Document(metadata={'source': 'https://www.langchain.com/', 'title': 'LangChain: Observe, Evaluate, and Deploy Reliable AI Agents', 'description': 'LangChain provides the engineering platform and open source frameworks developers use to build, test, and deploy reliable AI agents.', 'language': 'en'}, page_content='LangChain: Observe, Evaluate, and Deploy Reliable AI Agents\n\n\n\n\n\n\n\n\n\n\n\n\n\n\nJoin us May 13th &\xa0May 14th at Interrupt, the Agent Conference by LangChainBuy tickets\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\nProducts\n\nLangSmith Platform\n\nObservabilitySee exactly what your agents are doingEvaluationScore and improve agent performanceDeploymentShip and scale agents in productionFleetAgents for the whole companyOpen Source FrameworksdeepagentsBuild long-running agents for complex taskslangchainQuick start agents with any model providerlanggraphBuild reliable agents with low-level controlLearn\n\nResourcesBlogCustomer StoriesGuidesMax AgencyHow-T

## Séparer les text en chunks

In [17]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

loader = TextLoader("./test.txt") # or any other loader
docs = loader.load()

splitter = RecursiveCharacterTextSplitter(
    chunk_size=50, # le nb de de caractère dans un chunk
    chunk_overlap=10, # taille overlap, ie le chunk suivant comment par nb de mots de la fin de chunk précédent pour raison de context
    )

splitted_docs = splitter.split_documents(docs)
splitted_docs

[Document(metadata={'source': './test.txt'}, page_content="Bonjour, aujourd'hui est d'un beau jour avec le"),
 Document(metadata={'source': './test.txt'}, page_content='avec le gros soleil.'),
 Document(metadata={'source': './test.txt'}, page_content='Les choses les plus mystÃ¨re du monde sont des'),
 Document(metadata={'source': './test.txt'}, page_content='sont des mystÃ¨res.'),
 Document(metadata={'source': './test.txt'}, page_content='vous saurez ces secrets quand vous les'),
 Document(metadata={'source': './test.txt'}, page_content='vous les dÃ©couvrez.')]

In [7]:
from langchain_text_splitters import (
            Language,
            RecursiveCharacterTextSplitter,
            )
PYTHON_CODE = """
    def hello_world():
    print("Hello, World!")
    # Call the function
    hello_world()
    """

python_splitter = RecursiveCharacterTextSplitter.from_language(
    language=Language.PYTHON, chunk_size=50, chunk_overlap=0
    )

python_docs = python_splitter.create_documents([PYTHON_CODE])
python_docs

[Document(metadata={}, page_content='def hello_world():\n    print("Hello, World!")'),
 Document(metadata={}, page_content='# Call the function\n    hello_world()')]

In [6]:
markdown_text = """
# LangChain
⚡ Building applications with LLMs through composability ⚡
## Quick Install
```bash
pip install langchain
```
As an open source project in a rapidly developing field, we are extremely open
to contributions.
"""
md_splitter = RecursiveCharacterTextSplitter.from_language(
language=Language.MARKDOWN, chunk_size=60, chunk_overlap=0
)
md_docs = md_splitter.create_documents([markdown_text],
[{"source": "https://www.langchain.com"}])

md_docs

[Document(metadata={'source': 'https://www.langchain.com'}, page_content='# LangChain'),
 Document(metadata={'source': 'https://www.langchain.com'}, page_content='⚡ Building applications with LLMs through composability ⚡'),
 Document(metadata={'source': 'https://www.langchain.com'}, page_content='## Quick Install\n```bash\npip install langchain'),
 Document(metadata={'source': 'https://www.langchain.com'}, page_content='```'),
 Document(metadata={'source': 'https://www.langchain.com'}, page_content='As an open source project in a rapidly developing field, we'),
 Document(metadata={'source': 'https://www.langchain.com'}, page_content='are extremely open'),
 Document(metadata={'source': 'https://www.langchain.com'}, page_content='to contributions.')]

D'ici, on charge le text embeding le en vecteur, puis les stocker en local avec chrromdb.

In [2]:
from langchain_community.document_loaders import TextLoader
from langchain_community.embeddings import OllamaEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import Chroma

raw_documents = TextLoader('./test.txt').load()
text_splitter = RecursiveCharacterTextSplitter(chunk_size=50, chunk_overlap=20)
documents = text_splitter.split_documents(raw_documents)

embeddings_model = OllamaEmbeddings(model="llama3")

# Sauvegarde sur disque, aucun serveur requis
db = Chroma.from_documents(
    documents,
    embeddings_model,
    persist_directory="./chroma_db"  # dossier local
)

In [5]:
result = db.similarity_search("soleil",k=3)

for doc in result:
    print(doc.page_content)
    print("---")

beau jour avec le gros soleil.
---
quand vous les dÃ©couvrez.
---
Bonjour, aujourd'hui est d'un beau jour avec le
---


In [6]:
import uuid
from langchain_core.documents import Document

ids = [str(uuid.uuid4()), str(uuid.uuid4())]
db.add_documents(
    [
    Document(
    page_content="there are cats in the pond",
    metadata={"location": "pond", "topic": "animals"},
    ),
    Document(
    page_content="ducks are also found in the pond",
    metadata={"location": "pond", "topic": "animals"},
    ),
    ],
    ids=ids,
)

['6a90cdce-c384-4a8c-b5fa-2d22812c3fda',
 '85d2e2a1-4041-44a4-9863-58096b10af50']

In [7]:
db.add_documents(
    [
        Document(
            page_content="les ennuis sont quand on fait rien et ne pas s'amuser."
        )
    ]
)

['485e8bb0-8b66-4480-be5d-57853a81867c']

## La metadata

C'est des infos supplémentaires attachées à chaque document. Ce n'est **pas obligatoire**.

```python
# Sans metadata → ça marche très bien
Document(page_content="there are cats in the pond")

# Avec metadata → infos en bonus
Document(
    page_content="there are cats in the pond",
    metadata={"location": "pond", "topic": "animals"}
)
```

### À quoi ça sert ?

Ça permet de **filtrer** les résultats de recherche :

```python
# Sans filtre → cherche dans TOUS les documents
db.similarity_search("animals", k=4)

# Avec filtre → cherche UNIQUEMENT dans les docs du pond
db.similarity_search("animals", k=4, filter={"location": "pond"})
```

### Exemple concret

Imagine que tu indexes plein de fichiers différents :

```python
Document(page_content="...", metadata={"source": "rapport2023.pdf", "page": 1})
Document(page_content="...", metadata={"source": "rapport2024.pdf", "page": 1})
```

Tu peux chercher uniquement dans un fichier précis :

```python
db.similarity_search("chiffre d'affaires", filter={"source": "rapport2024.pdf"})
```

> Sans metadata tu peux quand même faire de la recherche sémantique,
> tu perds juste la possibilité de filtrer.

In [ ]:
retrieve = db.as_retriever()

docs = retrieve.invoke("""c'est quoi les ennuis?""")
docs

[Document(metadata={'source': './test.txt'}, page_content='quand vous les dÃ©couvrez.'),
 Document(metadata={'source': './test.txt'}, page_content='beau jour avec le gros soleil.'),
 Document(metadata={'source': './test.txt'}, page_content='du monde sont des mystÃ¨res.'),
 Document(metadata={}, page_content="les ennuis sont quand on fait rien et ne pas s'amuser.")]

In [ ]:
from langchain_community.chat_models import ChatOllama
from langchain_core.prompts import ChatPromptTemplate

retriever = db.as_retriever()

prompt = ChatPromptTemplate.from_template("""Answer the question based only on
the following context:
{context}

Question: {question}
""")

llm = ChatOllama(model="llama3", temperature=0)

chain = prompt | llm

docs = retriever.invoke("""c'est quoi les ennuis?""")

chain.invoke({
    "context": docs,
    "question": "c'est quoi les ennuis?"
})

AIMessage(content='According to the context, "les ennuis" are when on fait rien et ne pas s\'amuser.', additional_kwargs={}, response_metadata={'model': 'llama3', 'created_at': '2026-04-19T18:41:13.8604737Z', 'message': {'role': 'assistant', 'content': ''}, 'done': True, 'done_reason': 'stop', 'total_duration': 20376905700, 'load_duration': 238948300, 'prompt_eval_count': 122, 'prompt_eval_duration': 15689255100, 'eval_count': 24, 'eval_duration': 4393981900}, id='lc_run--019da70b-a4e7-7f03-a67a-5991547b5442-0', tool_calls=[], invalid_tool_calls=[])